# V14: Mask R-CNN + ResNet18 crop refiner, 15 detector epochs, candidate voting with the public recipe as the fallback

Rebuilds the architecture behind the public 0.70 leaderboard notebooks, with a test of the instance count.

* **Detector:** torchvision Mask R-CNN ResNet50-FPN **V2** on the whole frame at 1024 px, anchors 16-256, SGD lr 0.005 with
  drops at 70% and 90% of the epochs, two GPUs, batch 1 per GPU. Trained on **one annotator set per photo** (the one with the most
  filaments), as in the public notebooks.
* **Refiner:** ResNet18 U-Net on 256 px square crops (1.8x the box), BCE + soft Dice, cosine schedule.
* **Epochs:** 15 for the detector and 10 for the refiner. The public notebook uses 10 (validation run) and then 6-10 (final run
  on every photo), and 8 refiner epochs then 5-8. Its per-epoch minutes are not published (the saved notebook has no output).
* **Inference:** 4 flipped views of the frame, **up to 10 candidates per view**, instances grouped across views by IoU >= 0.5, then
  **voted down**: clusters seen in fewer than `min_support` views are dropped and only the best `cap` (7-8 is the level the top
  scores sit at, about 7.2 per photo) are kept. Each is refined with 4 flipped crops, thresholded, reduced to its largest piece,
  and overlapping masks are merged (a filament found twice becomes one).
* **Tuning:** the 20% validation photos pick `cap`, `min_support` and the refiner threshold on the soft instance Dice used by the
  public scorers. The table also prints PQ and the mean instances per photo for every cap.

Credit: Nomannic19 (Mask R-CNN + U-Net Refiner) and Fususu (flip test-time views, refiner threshold, overlap merge). The voting
and the instance cap are new.


In [ ]:
%pip install -q pycocotools scipy 'opencv-python-headless>=4.9,<5'

In [ ]:
import json
import os
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import torch
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO

VERSION = "V14"
CFG = dict(
    seed=2026,
    detector_epochs=15, detector_lr=0.005,
    refiner_epochs=10, refiner_batch=16,
    candidate_threshold=0.2, max_candidates=10,     # up to 10 candidates per view
    caps=[7, 8, 10, None], min_supports=[1, 2, 3],  # vote down to the cap; None keeps every voted cluster
    min_scores=[0.2, 0.5, 0.7],                     # floor on a candidate's mean detector score
    refiner_thresholds=[0.7, 0.85],
    select_metric="blend",                          # "blend" = mean of soft Dice and PQ; or "soft_dice" / "pq"
    recipe_margin=0.01,                             # voted settings must beat the public recipe by this much
)
COMPETITION = "filament-segmentation-2026"
DATASET_NAME = "MAGFiLO_1.0_Kaggle_2026"
ANNOTATION_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
DATA_ROOT_OVERRIDE = None
WORK_DIR = (Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()) / "v13_work"
WORK_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = WORK_DIR.parent / "submission.csv"
random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
GPUS = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(GPUS)],
      "| free disk: %.1f GB" % (shutil.disk_usage(WORK_DIR).free / 1e9))
if GPUS < 1:
    print("WARNING: no GPU visible; training will not run. Use a T4 x2 session.")


## Data, split and library files
One annotator set per photo, 80/20 split stratified by site and filament count (as in the public notebook).

In [ ]:
LIB_SOURCE = '"""V13: Mask R-CNN detector + ResNet18 crop U-Net refiner with candidate voting.\n\nThe detector and refiner follow the public "Mask R-CNN + U-Net Refiner" notebook by Nomannic19 and the\n"LB 0.70 Topology-Safe Solar Filaments" notebook by Fususu (flip test-time views, refiner threshold, positive-overlap merge).\nThe candidate voting (predict up to 10 per view, vote down to a cap) is new here.\n"""\n\nimport json\nimport random\nimport time\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom pycocotools import mask as mask_utils\nfrom pycocotools.coco import COCO\nfrom scipy import ndimage as ndi\nfrom scipy.optimize import linear_sum_assignment\nfrom torch import nn\nfrom torch.utils.data import Dataset\n\nIMAGE_SIZE = 1024\nFLIPS = [(), (1,), (0,), (0, 1)]  # identity, horizontal, vertical, both (axes of a 2-D frame)\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef encode_mask(mask):\n    return mask_utils.encode(np.asfortranarray(np.asarray(mask).astype(np.uint8)))\n\n\ndef largest_component(mask):\n    labels, count = ndi.label(mask, structure=np.ones((3, 3), dtype=np.uint8))\n    if count <= 1:\n        return mask\n    return labels == (np.argmax(np.bincount(labels.ravel())[1:]) + 1)\n\n\ndef merge_overlapping_rles(rles):\n    """Union every group of instances that overlap by at least one pixel (removes duplicate detections)."""\n    if len(rles) < 2:\n        return list(rles)\n    adjacency = mask_utils.iou(rles, rles, [0] * len(rles)) > 0\n    groups, seen = [], set()\n    for start in range(len(rles)):\n        if start in seen:\n            continue\n        stack, group = [start], []\n        while stack:\n            index = stack.pop()\n            if index in seen:\n                continue\n            seen.add(index)\n            group.append(index)\n            stack.extend(np.flatnonzero(adjacency[index]).tolist())\n        groups.append(group)\n    return [mask_utils.merge([rles[i] for i in group]) if len(group) > 1 else rles[group[0]] for group in groups]\n\n\ndef square_bounds(mask, context=1.8, minimum=96):\n    height, width = mask.shape\n    ys, xs = np.where(mask)\n    center_x, center_y = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2\n    side = int(np.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))\n    side = min(max(side, minimum), min(height, width))\n    x0, y0 = int(round(center_x - side / 2)), int(round(center_y - side / 2))\n    x1, y1 = x0 + side, y0 + side\n    if x0 < 0:\n        x1 -= x0\n        x0 = 0\n    if y0 < 0:\n        y1 -= y0\n        y0 = 0\n    if x1 > width:\n        x0 -= x1 - width\n        x1 = width\n    if y1 > height:\n        y0 -= y1 - height\n        y1 = height\n    return int(x0), int(y0), int(x1), int(y1)\n\n\n# ----------------------------------------------------------------------------------------------- manifest\n\ndef build_manifest(annotation_path, seed, val_fraction=0.2):\n    """One annotator set per physical observation (the one with the most filaments), stratified 80/20."""\n    coco = COCO(str(annotation_path))\n    best = {}\n    for image_id, info in coco.imgs.items():\n        annotations = coco.imgToAnns[image_id]\n        key = (len(annotations), sum(float(a["area"]) for a in annotations), str(image_id))\n        stem = Path(info["file_name"]).stem\n        if stem not in best or key > best[stem][0]:\n            best[stem] = (key, dict(id=str(image_id), file_name=info["file_name"], observation_id=stem,\n                                    site=stem[-2:], n_filaments=len(annotations)))\n    rows = [value[1] for value in best.values()]\n    rows.sort(key=lambda row: row["file_name"])\n    counts = np.array([row["n_filaments"] for row in rows])\n    edges = np.unique(np.quantile(counts, [0.25, 0.5, 0.75]))\n    strata = {}\n    for row in rows:\n        row["count_bin"] = int(np.searchsorted(edges, row["n_filaments"], side="right"))\n        strata.setdefault((row["site"], row["count_bin"]), []).append(row)\n    rng = random.Random(seed)\n    for members in strata.values():\n        rng.shuffle(members)\n        take = int(round(len(members) * val_fraction)) if len(members) > 1 else 0\n        for index, row in enumerate(members):\n            row["split"] = "val" if index < take else "train"\n    return rows\n\n\ndef all_annotator_ids(annotation_path):\n    """observation_id -> every annotator record id (used to score against all human sets)."""\n    coco = COCO(str(annotation_path))\n    groups = {}\n    for image_id, info in coco.imgs.items():\n        groups.setdefault(Path(info["file_name"]).stem, []).append(str(image_id))\n    return groups\n\n\n# ----------------------------------------------------------------------------------------------- detector\n\ndef masks_to_boxes(masks):\n    boxes = []\n    for mask in masks:\n        ys, xs = np.where(mask)\n        boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])\n    return np.asarray(boxes, dtype=np.float32).reshape(-1, 4)\n\n\nclass FilamentDataset(Dataset):\n    def __init__(self, records, annotation_path, image_dir, augment=False):\n        self.records, self.augment = list(records), augment\n        self.image_dir = Path(image_dir)\n        self.coco = COCO(str(annotation_path))\n\n    def __len__(self):\n        return len(self.records)\n\n    @staticmethod\n    def photometric(image):\n        valid = image > 0.01\n        if np.random.rand() < 0.70:\n            image[valid] = image[valid] ** np.random.uniform(0.80, 1.25)\n        if np.random.rand() < 0.70:\n            mean = image[valid].mean()\n            contrast, gain = np.random.uniform(0.85, 1.15), np.random.uniform(0.90, 1.10)\n            image[valid] = ((image[valid] - mean) * contrast + mean) * gain\n        if np.random.rand() < 0.15:\n            blurred = cv2.GaussianBlur(image, (0, 0), np.random.uniform(0.30, 0.80))\n            image[valid] = blurred[valid]\n        if np.random.rand() < 0.25:\n            image[valid] += np.random.normal(0, np.random.uniform(0.003, 0.015), valid.sum())\n        return np.clip(image, 0, 1)\n\n    def __getitem__(self, index):\n        row = self.records[index]\n        image = cv2.imread(str(self.image_dir / row["file_name"]), cv2.IMREAD_GRAYSCALE)\n        image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32) / 255\n        masks = []\n        for annotation in self.coco.imgToAnns[row["id"]]:\n            mask = cv2.resize(self.coco.annToMask(annotation), (IMAGE_SIZE, IMAGE_SIZE),\n                              interpolation=cv2.INTER_NEAREST).astype(bool)\n            if mask.any():\n                masks.append(mask)\n        masks = np.stack(masks) if masks else np.zeros((0, IMAGE_SIZE, IMAGE_SIZE), bool)\n        if self.augment:\n            rotations = np.random.randint(4)\n            image = np.rot90(image, rotations)\n            masks = np.rot90(masks, rotations, axes=(1, 2))\n            if np.random.rand() < 0.5:\n                image = np.flip(image, axis=1)\n                masks = np.flip(masks, axis=2)\n            image = self.photometric(np.ascontiguousarray(image))\n        image, masks = np.ascontiguousarray(image), np.ascontiguousarray(masks)\n        target = dict(\n            boxes=torch.as_tensor(masks_to_boxes(masks), dtype=torch.float32),\n            labels=torch.ones(len(masks), dtype=torch.int64),\n            masks=torch.as_tensor(masks, dtype=torch.uint8),\n            image_id=torch.tensor([index]),\n            area=torch.as_tensor(masks.sum((1, 2)), dtype=torch.float32),\n            iscrowd=torch.zeros(len(masks), dtype=torch.int64))\n        return torch.from_numpy(image).unsqueeze(0).repeat(3, 1, 1), target, row["observation_id"]\n\n\ndef collate_fn(batch):\n    return tuple(zip(*batch))\n\n\ndef build_detector(pretrained=True):\n    from torchvision.models.detection import MaskRCNN_ResNet50_FPN_V2_Weights, maskrcnn_resnet50_fpn_v2\n    from torchvision.models.detection.anchor_utils import AnchorGenerator\n    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor\n    from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor\n    from torchvision.ops import MultiScaleRoIAlign\n\n    weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT if pretrained else None\n    model = maskrcnn_resnet50_fpn_v2(weights=weights, weights_backbone=None)\n    box_features = model.roi_heads.box_predictor.cls_score.in_features\n    mask_features = model.roi_heads.mask_predictor.conv5_mask.in_channels\n    hidden = model.roi_heads.mask_predictor.conv5_mask.out_channels\n    model.roi_heads.box_predictor = FastRCNNPredictor(box_features, 2)\n    model.roi_heads.mask_predictor = MaskRCNNPredictor(mask_features, hidden, 2)\n    model.rpn.anchor_generator = AnchorGenerator(\n        sizes=((16,), (32,), (64,), (128,), (256,)), aspect_ratios=((0.5, 1.0, 2.0),) * 5)\n    model.roi_heads.mask_roi_pool = MultiScaleRoIAlign(\n        featmap_names=["0", "1", "2", "3"], output_size=28, sampling_ratio=2)\n    model.transform.min_size, model.transform.max_size = (IMAGE_SIZE,), IMAGE_SIZE\n    return model\n\n\n# ----------------------------------------------------------------------------------------------- scoring\n\ndef instance_scores(predicted, truth):\n    """Soft instance Dice (matched Dice / max(p, t), as in the public scorers) and PQ (IoU > 0.5)."""\n    p, t = len(predicted), len(truth)\n    if not p or not t:\n        both_empty = float(p == 0 and t == 0)\n        return dict(soft_dice=both_empty, pq=both_empty, tp=0, predictions=p)\n    iou = mask_utils.iou(predicted, truth, [0] * t)\n    dice = 2 * iou / (1 + iou + 1e-7)\n    rows, columns = linear_sum_assignment(-dice)\n    soft = float(dice[rows, columns].sum() / max(p, t))\n    valid = iou > 0.5\n    rewards = valid * (1 + iou / (min(p, t) + 1))\n    rows, columns = linear_sum_assignment(-rewards)\n    matched = [float(iou[r, c]) for r, c in zip(rows, columns) if valid[r, c]]\n    tp = len(matched)\n    denominator = tp + 0.5 * (p - tp) + 0.5 * (t - tp)\n    return dict(soft_dice=soft, pq=sum(matched) / denominator if denominator else 1.0, tp=tp, predictions=p)\n\n\ndef mean_scores(predictions, references):\n    """predictions: obs -> [rle]; references: obs -> [[rle] per annotator set]. Averages sets, then images."""\n    rows = []\n    for observation, sets in references.items():\n        per_set = [instance_scores(predictions.get(observation, []), truth) for truth in sets]\n        rows.append({key: float(np.mean([s[key] for s in per_set])) for key in per_set[0]})\n    return {key: float(np.mean([row[key] for row in rows])) for key in rows[0]}\n\n\ndef annotation_rles(coco, record_id, size):\n    rles = []\n    for annotation in coco.imgToAnns[str(record_id)]:\n        mask = coco.annToMask(annotation)\n        if mask.shape != (size, size):\n            mask = cv2.resize(mask, (size, size), interpolation=cv2.INTER_NEAREST)\n        if mask.any():\n            rles.append(encode_mask(mask))\n    return rles\n\n\n# ----------------------------------------------------------------------------------------------- detector loop\n\ndef train_one_epoch(model, loader, optimizer, device, scaler=None):\n    model.train()\n    total, count = 0.0, 0\n    for images, targets, _ in loader:\n        images = [image.to(device, non_blocking=True) for image in images]\n        targets = [{k: v.to(device, non_blocking=True) for k, v in target.items()} for target in targets]\n        optimizer.zero_grad(set_to_none=True)\n        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=scaler is not None):\n            loss = sum(model(images, targets).values())\n        if scaler is not None:\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n        else:\n            loss.backward()\n            optimizer.step()\n        total += float(loss.detach())\n        count += 1\n    return total / max(count, 1)\n\n\n@torch.inference_mode()\ndef evaluate_detector(model, loader, device, score_threshold=0.6, mask_threshold=0.5):\n    model.eval()\n    rows = []\n    for images, targets, _ in loader:\n        output = model([images[0].to(device)])[0]\n        keep = output["scores"] >= score_threshold\n        predicted = [encode_mask(m) for m in (output["masks"][keep, 0] >= mask_threshold).cpu().numpy()]\n        predicted = merge_overlapping_rles([r for r in predicted if mask_utils.area(r) > 0])\n        truth = [encode_mask(m) for m in targets[0]["masks"].numpy()]\n        rows.append(instance_scores(predicted, truth))\n    return {key: float(np.mean([row[key] for row in rows])) for key in rows[0]}\n\n\n# ----------------------------------------------------------------------------------------------- refiner\n\nclass CropDataset(Dataset):\n    def __init__(self, image_path, mask_path, augment=False):\n        self.images = np.load(image_path, mmap_mode="r")\n        self.masks = np.load(mask_path, mmap_mode="r")\n        self.augment = augment\n\n    def __len__(self):\n        return len(self.images)\n\n    def __getitem__(self, index):\n        image = torch.from_numpy(np.asarray(self.images[index]).copy()).float() / 255\n        mask = torch.from_numpy(np.asarray(self.masks[index]).copy()).float().unsqueeze(0)\n        if self.augment:\n            rotations = random.randrange(4)\n            image, mask = torch.rot90(image, rotations, (0, 1)), torch.rot90(mask, rotations, (1, 2))\n            if random.random() < 0.5:\n                image, mask = torch.flip(image, (1,)), torch.flip(mask, (2,))\n            if random.random() < 0.5:\n                image = image ** random.uniform(0.85, 1.20)\n            if random.random() < 0.5:\n                image = torch.clamp(image * random.uniform(0.90, 1.10), 0, 1)\n        return image.unsqueeze(0).repeat(3, 1, 1), mask\n\n\nclass ConvBlock(nn.Sequential):\n    def __init__(self, in_channels, out_channels):\n        super().__init__(\n            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False), nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True))\n\n\nclass CropUNet(nn.Module):\n    """ResNet18 encoder U-Net on 256 x 256 instance crops."""\n\n    def __init__(self, pretrained=True):\n        super().__init__()\n        from torchvision.models import ResNet18_Weights, resnet18\n        encoder = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)\n        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)\n        self.pool, self.layer1, self.layer2 = encoder.maxpool, encoder.layer1, encoder.layer2\n        self.layer3, self.layer4 = encoder.layer3, encoder.layer4\n        self.up4, self.dec4 = nn.ConvTranspose2d(512, 256, 2, 2), ConvBlock(512, 256)\n        self.up3, self.dec3 = nn.ConvTranspose2d(256, 128, 2, 2), ConvBlock(256, 128)\n        self.up2, self.dec2 = nn.ConvTranspose2d(128, 64, 2, 2), ConvBlock(128, 64)\n        self.up1, self.dec1 = nn.ConvTranspose2d(64, 32, 2, 2), ConvBlock(96, 32)\n        self.up0, self.head = nn.ConvTranspose2d(32, 16, 2, 2), nn.Conv2d(16, 1, 1)\n        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))\n        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))\n\n    def forward(self, image):\n        image = (image - self.mean) / self.std\n        x0 = self.stem(image)\n        x1 = self.layer1(self.pool(x0))\n        x2 = self.layer2(x1)\n        x3 = self.layer3(x2)\n        x4 = self.layer4(x3)\n        x = self.dec4(torch.cat([self.up4(x4), x3], 1))\n        x = self.dec3(torch.cat([self.up3(x), x2], 1))\n        x = self.dec2(torch.cat([self.up2(x), x1], 1))\n        x = self.dec1(torch.cat([self.up1(x), x0], 1))\n        return self.head(self.up0(x))\n\n\ndef soft_dice_loss(logits, targets):\n    probabilities = logits.sigmoid()\n    intersection = (probabilities * targets).sum((1, 2, 3))\n    denominator = probabilities.sum((1, 2, 3)) + targets.sum((1, 2, 3))\n    return (1 - (2 * intersection + 1) / (denominator + 1)).mean()\n\n\ndef build_crop_cache(coco, rows, image_dir, directory, prefix, size=256):\n    directory = Path(directory)\n    total = sum(len(coco.imgToAnns[row["id"]]) for row in rows)\n    image_path, mask_path = directory / f"{prefix}_crop_images.npy", directory / f"{prefix}_crop_masks.npy"\n    images = np.lib.format.open_memmap(image_path, mode="w+", dtype=np.uint8, shape=(total, size, size))\n    masks = np.lib.format.open_memmap(mask_path, mode="w+", dtype=np.uint8, shape=(total, size, size))\n    position = 0\n    for row in rows:\n        image = cv2.imread(str(Path(image_dir) / row["file_name"]), cv2.IMREAD_GRAYSCALE)\n        for annotation in coco.imgToAnns[row["id"]]:\n            mask = coco.annToMask(annotation).astype(np.uint8)\n            x0, y0, x1, y1 = square_bounds(mask)\n            images[position] = cv2.resize(image[y0:y1, x0:x1], (size, size), interpolation=cv2.INTER_LINEAR)\n            masks[position] = cv2.resize(mask[y0:y1, x0:x1], (size, size), interpolation=cv2.INTER_NEAREST)\n            position += 1\n    images.flush()\n    masks.flush()\n    return image_path, mask_path, total\n\n\ndef train_refiner(model, train_loader, val_loader, epochs, output_path, device, log=print):\n    model.to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)\n    amp = device.type == "cuda"\n    scaler = torch.amp.GradScaler("cuda", enabled=amp)\n    history, best = [], -1.0\n    for epoch in range(1, epochs + 1):\n        started = time.perf_counter()\n        model.train()\n        total = samples = 0\n        for images, masks in train_loader:\n            images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):\n                logits = model(images)\n                loss = 0.5 * nn.functional.binary_cross_entropy_with_logits(logits, masks) \\\n                    + 0.5 * soft_dice_loss(logits, masks)\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n            total += loss.item() * len(images)\n            samples += len(images)\n        model.eval()\n        dice_total = count = 0\n        with torch.inference_mode():\n            for images, masks in val_loader:\n                masks = masks.to(device).bool()\n                predictions = model(images.to(device)).sigmoid() >= 0.65\n                intersection = (predictions & masks).sum((1, 2, 3)).float()\n                denominator = predictions.sum((1, 2, 3)) + masks.sum((1, 2, 3))\n                dice_total += ((2 * intersection + 1e-7) / (denominator + 1e-7)).sum().item()\n                count += len(images)\n        val_dice = dice_total / max(count, 1)\n        if val_dice > best:\n            best = val_dice\n            torch.save({"epoch": epoch, "model": model.state_dict()}, output_path)\n        scheduler.step()\n        history.append(dict(epoch=epoch, loss=total / samples, val_dice=val_dice,\n                            minutes=(time.perf_counter() - started) / 60))\n        log(f"REFINER {epoch:02d}/{epochs} | loss {history[-1][\'loss\']:.4f} | val Dice {val_dice:.4f} "\n            f"| {history[-1][\'minutes\']:.2f} min")\n    return history\n\n\n# ----------------------------------------------------------------------------------------------- candidates + voting\n\ndef flip_array(array, axes, offset=0):\n    return np.ascontiguousarray(np.flip(array, axis=tuple(a + offset for a in axes))) if axes else array\n\n\n@torch.inference_mode()\ndef detect_views(detector, image, device, score_threshold, max_candidates, mask_threshold=0.5, flips=FLIPS):\n    """Run the detector on flipped copies; each view keeps its top `max_candidates` detections."""\n    detector.roi_heads.score_thresh = score_threshold\n    detector.roi_heads.detections_per_img = max_candidates\n    views = []\n    for axes in flips:\n        tensor = torch.from_numpy(flip_array(image, axes)).unsqueeze(0).repeat(3, 1, 1).to(device)\n        output = detector([tensor])[0]\n        masks = (output["masks"][:, 0] >= mask_threshold).cpu().numpy()\n        scores = output["scores"].cpu().numpy()\n        rles, kept = [], []\n        for mask, score in zip(masks, scores):\n            mask = flip_array(mask, axes)\n            if mask.any():\n                rles.append(encode_mask(mask))\n                kept.append(float(score))\n        views.append((rles, kept))\n    return views\n\n\ndef cluster_views(views, match_iou=0.5):\n    """Group instances across views: one-to-one IoU matching, at most one member per view."""\n    clusters = []\n    for view_index, (rles, scores) in enumerate(views):\n        if not rles:\n            continue\n        matched = set()\n        if clusters:\n            overlaps = mask_utils.iou([c["rep"] for c in clusters], rles, [0] * len(rles))\n            eligible = overlaps >= match_iou\n            weights = eligible * (min(overlaps.shape) + 1 + overlaps)\n            for row, column in zip(*linear_sum_assignment(-weights)):\n                if eligible[row, column]:\n                    clusters[row]["members"].append(rles[column])\n                    clusters[row]["scores"].append(scores[column])\n                    matched.add(column)\n        clusters.extend(dict(rep=rle, members=[rle], scores=[scores[column]], origin=view_index)\n                        for column, rle in enumerate(rles) if column not in matched)\n    return clusters\n\n\ndef fuse_cluster(cluster, full_size):\n    """Majority-vote mask of one cluster, resized to the full frame."""\n    members = cluster["members"]\n    votes = np.zeros(mask_utils.decode(members[0]).shape, dtype=np.uint8)\n    for rle in members:\n        votes += mask_utils.decode(rle)\n    fused = votes >= (len(members) + 1) // 2\n    if not fused.any():\n        fused = votes > 0\n    return cv2.resize(fused.astype(np.uint8), (full_size[1], full_size[0]), interpolation=cv2.INTER_NEAREST)\n\n\ndef square_crop(image, mask, size=256):\n    x0, y0, x1, y1 = square_bounds(mask)\n    return (x0, y0, x1, y1), cv2.resize(image[y0:y1, x0:x1], (size, size), interpolation=cv2.INTER_LINEAR)\n\n\n@torch.inference_mode()\ndef refiner_probabilities(refiner, crops, device, batch=32, flips=FLIPS):\n    """Average the refiner\'s probability over the flipped copies of each 256 x 256 crop."""\n    out = []\n    for start in range(0, len(crops), batch):\n        tensor = torch.from_numpy(np.stack(crops[start:start + batch])).float().unsqueeze(1).repeat(1, 3, 1, 1)\n        tensor = tensor.to(device) / 255\n        total = 0\n        for axes in flips:\n            dims = tuple(a + 2 for a in axes)\n            view = torch.flip(tensor, dims) if dims else tensor\n            probability = refiner(view).sigmoid()\n            total = total + (torch.flip(probability, dims) if dims else probability)\n        out.append((total / len(flips)).cpu().numpy()[:, 0].astype(np.float16))\n    return np.concatenate(out) if out else np.zeros((0, 256, 256), np.float16)\n\n\ndef image_candidates(detector, refiner, image_path, device, settings):\n    """All voted candidates of one photo: fused mask, refiner probability map and vote support."""\n    gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)\n    height, width = gray.shape\n    small = cv2.resize(gray, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32) / 255\n    views = detect_views(detector, small, device, settings["candidate_threshold"], settings["max_candidates"])\n    clusters = cluster_views(views)\n    fused = [fuse_cluster(c, (height, width)) for c in clusters]\n    crops, bounds = [], []\n    for mask in fused:\n        box, crop = square_crop(gray, mask)\n        bounds.append(box)\n        crops.append(crop)\n    probabilities = refiner_probabilities(refiner, crops, device) if refiner is not None and crops else []\n    candidates = []\n    for index, cluster in enumerate(clusters):\n        candidates.append(dict(\n            support=len(cluster["members"]), score=float(np.mean(cluster["scores"])),\n            origin=cluster["origin"], first_score=float(cluster["scores"][0]),\n            fallback=encode_mask(fused[index]), bounds=bounds[index], shape=(height, width),\n            probability=probabilities[index] if len(probabilities) else None, refined={}))\n    first = views[0]\n    return dict(candidates=candidates, view0=first, shape=(height, width))\n\n\ndef refined_rle(candidate, threshold, single_piece):\n    if candidate["probability"] is None:\n        return candidate["fallback"]\n    key = (threshold, single_piece)\n    if key not in candidate["refined"]:\n        x0, y0, x1, y1 = candidate["bounds"]\n        probability = cv2.resize(candidate["probability"].astype(np.float32), (x1 - x0, y1 - y0),\n                                 interpolation=cv2.INTER_LINEAR)\n        keep = probability >= threshold\n        if single_piece and keep.any():\n            keep = largest_component(keep)\n        if keep.any():\n            full = np.zeros(candidate["shape"], dtype=np.uint8)\n            full[y0:y1, x0:x1] = keep\n            candidate["refined"][key] = encode_mask(full)\n        else:\n            candidate["refined"][key] = candidate["fallback"]\n    return candidate["refined"][key]\n\n\ndef select_instances(item, cap, min_support, threshold, single_piece=True, min_area=64,\n                     min_score=0.0, view0_only=False):\n    """Vote down: drop clusters seen in too few views or with a low mean score, keep the best `cap`,\n    refine, merge duplicates. `view0_only` mimics the public recipe (identity view, score floor, no voting)."""\n    candidates = sorted(item["candidates"], key=lambda c: (c["support"], c["score"]), reverse=True)\n    if view0_only:\n        chosen = [c for c in candidates if c["origin"] == 0 and c["first_score"] >= min_score]\n    else:\n        chosen = [c for c in candidates if c["support"] >= min_support and c["score"] >= min_score]\n    if not chosen and candidates:\n        chosen = candidates[:1]\n    chosen = chosen[:cap] if cap else chosen\n    rles = [refined_rle(c, threshold, single_piece) for c in chosen]\n    rles = [r for r in rles if mask_utils.area(r) >= min_area] or rles[:1]\n    merged = merge_overlapping_rles(rles)\n    merged.sort(key=lambda r: (float(mask_utils.toBbox(r)[1]), float(mask_utils.toBbox(r)[0])))\n    return merged\n\n\ndef single_view_rles(item, score_threshold=0.6):\n    """Reference: one view, score >= threshold, detector masks only, overlaps merged (the public recipe)."""\n    rles, scores = item["view0"]\n    height, width = item["shape"]\n    out = []\n    for rle, score in zip(rles, scores):\n        if score >= score_threshold:\n            mask = cv2.resize(mask_utils.decode(rle), (width, height), interpolation=cv2.INTER_NEAREST)\n            out.append(encode_mask(mask))\n    return merge_overlapping_rles(out)\n\n\ndef save_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2))\n'
TRAIN_SOURCE = '"""Distributed Mask R-CNN training for V13 (run with torchrun on two GPUs)."""\n\nimport argparse\nimport json\nimport os\nimport sys\nimport time\nfrom pathlib import Path\n\nimport torch\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import DataLoader\nfrom torch.utils.data.distributed import DistributedSampler\nfrom tqdm.auto import tqdm\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport v13_lib as lib  # noqa: E402\n\n\ndef seed_worker(worker_id):\n    import random\n\n    import numpy as np\n    seed = torch.initial_seed() % 2**32\n    random.seed(seed)\n    np.random.seed(seed)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--annotations", required=True)\n    parser.add_argument("--images", required=True)\n    parser.add_argument("--work", required=True)\n    parser.add_argument("--epochs", type=int, default=10)\n    parser.add_argument("--seed", type=int, default=2026)\n    parser.add_argument("--lr", type=float, default=0.005)\n    args = parser.parse_args()\n    work = Path(args.work)\n\n    local_rank = int(os.environ["LOCAL_RANK"])\n    torch.cuda.set_device(local_rank)\n    device = torch.device("cuda", local_rank)\n    dist.init_process_group("nccl")\n    rank = dist.get_rank()\n    lib.seed_everything(args.seed + rank)\n    torch.backends.cudnn.benchmark = True\n\n    manifest = json.loads((work / "manifest.json").read_text())\n    train_rows = [r for r in manifest if r["split"] == "train"]\n    val_rows = [r for r in manifest if r["split"] == "val"]\n    train_dataset = lib.FilamentDataset(train_rows, args.annotations, args.images, augment=True)\n    sampler = DistributedSampler(train_dataset, shuffle=True, seed=args.seed)\n    options = dict(num_workers=2, pin_memory=True, persistent_workers=True,\n                   collate_fn=lib.collate_fn, worker_init_fn=seed_worker)\n    train_loader = DataLoader(train_dataset, batch_size=1, sampler=sampler, **options)\n    val_loader = None\n    if rank == 0:\n        val_loader = DataLoader(lib.FilamentDataset(val_rows, args.annotations, args.images),\n                                batch_size=1, shuffle=False, **options)\n\n    model = DDP(lib.build_detector(pretrained=True).to(device), device_ids=[local_rank], broadcast_buffers=False)\n    optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],\n                                lr=args.lr, momentum=0.9, weight_decay=1e-4)\n    milestones = sorted({max(1, round(args.epochs * 0.70)), max(2, round(args.epochs * 0.90))})\n    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=0.1)\n    scaler = torch.amp.GradScaler("cuda")\n    history, best, started_all = [], -1.0, time.perf_counter()\n\n    for epoch in range(1, args.epochs + 1):\n        sampler.set_epoch(epoch)\n        started = time.perf_counter()\n        model.train()\n        total = torch.zeros(2, device=device)\n        for images, targets, _ in tqdm(train_loader, desc=f"Epoch {epoch:02d}", disable=rank != 0, leave=False):\n            images = [image.to(device, non_blocking=True) for image in images]\n            targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast("cuda", dtype=torch.float16):\n                loss = sum(model(images, targets).values())\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n            total[0] += loss.detach()\n            total[1] += 1\n        dist.all_reduce(total)\n        train_seconds = time.perf_counter() - started\n        metrics = lib.evaluate_detector(model.module, val_loader, device) if rank == 0 else {}\n        dist.barrier(device_ids=[local_rank])\n        if rank == 0:\n            row = dict(epoch=epoch, train_loss=(total[0] / total[1]).item(), lr=optimizer.param_groups[0]["lr"],\n                       train_minutes=train_seconds / 60, minutes=(time.perf_counter() - started) / 60, **metrics)\n            history.append(row)\n            if metrics["soft_dice"] > best:\n                best = metrics["soft_dice"]\n                torch.save({"epoch": epoch, "model": model.module.state_dict()}, work / "best_detector.pt")\n            print(f"DETECTOR {epoch:02d}/{args.epochs} | loss {row[\'train_loss\']:.4f} | val soft Dice "\n                  f"{metrics[\'soft_dice\']:.4f} | val PQ {metrics[\'pq\']:.4f} | {metrics[\'predictions\']:.1f} masks/img "\n                  f"| train {row[\'train_minutes\']:.2f} min | epoch {row[\'minutes\']:.2f} min", flush=True)\n        scheduler.step()\n        dist.barrier(device_ids=[local_rank])\n\n    if rank == 0:\n        torch.save({"epoch": args.epochs, "model": model.module.state_dict()}, work / "last_detector.pt")\n        (work / "detector_history.json").write_text(json.dumps(history, indent=2))\n        print(f"DETECTOR total {(time.perf_counter() - started_all) / 60:.1f} min", flush=True)\n    dist.destroy_process_group()\n\n\nif __name__ == "__main__":\n    main()\n'
candidates = ([Path(DATA_ROOT_OVERRIDE)] if DATA_ROOT_OVERRIDE else [
    Path("/kaggle/input/competitions") / COMPETITION / DATASET_NAME,
    Path("/kaggle/input") / COMPETITION / DATASET_NAME,
    Path.cwd() / "data" / DATASET_NAME, Path.cwd() / DATASET_NAME])
if DATA_ROOT_OVERRIDE is None and Path("/kaggle/input").is_dir():
    candidates += sorted(Path("/kaggle/input").glob(f"*/{DATASET_NAME}"))
    candidates += sorted(Path("/kaggle/input").glob(f"*/*/{DATASET_NAME}"))
valid_roots = [p for p in candidates if (p / "train" / ANNOTATION_NAME).is_file() and (p / "test" / "test_images").is_dir()]
if not valid_roots:
    raise FileNotFoundError("Attach the competition data or set DATA_ROOT_OVERRIDE.")
DATA_ROOT = valid_roots[0]
ANNOTATIONS = DATA_ROOT / "train" / ANNOTATION_NAME
TRAIN_DIR, TEST_DIR = DATA_ROOT / "train" / "train_images", DATA_ROOT / "test" / "test_images"
test_files = sorted(TEST_DIR.glob("*.jpeg"))

(WORK_DIR / "v13_lib.py").write_text(LIB_SOURCE)
(WORK_DIR / "v13_train_detector.py").write_text(TRAIN_SOURCE)
sys.path.insert(0, str(WORK_DIR))
import v13_lib as lib

manifest = lib.build_manifest(ANNOTATIONS, CFG["seed"])
(WORK_DIR / "manifest.json").write_text(json.dumps(manifest))
train_rows = [r for r in manifest if r["split"] == "train"]
val_rows = [r for r in manifest if r["split"] == "val"]
print(f"Photos: {len(manifest)} (train {len(train_rows)}, validation {len(val_rows)}), test {len(test_files)}")
print("Training filaments per photo (annotator set with the most filaments): "
      f"{np.mean([r['n_filaments'] for r in train_rows]):.2f}")

from torchvision.models import ResNet18_Weights
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_V2_Weights
try:  # download once here so the two training processes do not race on the cache
    MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT.get_state_dict(progress=False, check_hash=True)
    ResNet18_Weights.DEFAULT.get_state_dict(progress=False, check_hash=True)
    print("Pretrained weights cached.")
except Exception as error:
    print("WARNING: pretrained weights did not download:", error)


## Detector (15 epochs, two GPUs)
Prints validation soft Dice, PQ, masks per photo and the minutes of every epoch.

In [ ]:
def run_streaming(command):
    env = dict(os.environ, TQDM_DISABLE="1", PYTHONUNBUFFERED="1")
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in process.stdout:
        if line.strip():
            print(line.rstrip(), flush=True)
    return process.wait()


started = time.monotonic()
code = run_streaming([sys.executable, "-m", "torch.distributed.run", "--standalone",
                      f"--nproc_per_node={max(1, min(2, GPUS))}", str(WORK_DIR / "v13_train_detector.py"),
                      "--annotations", str(ANNOTATIONS), "--images", str(TRAIN_DIR), "--work", str(WORK_DIR),
                      "--epochs", str(CFG["detector_epochs"]), "--seed", str(CFG["seed"]),
                      "--lr", str(CFG["detector_lr"])])
print("detector exit code:", code, f"| {(time.monotonic() - started) / 60:.1f} min")
history_path = WORK_DIR / "detector_history.json"
if history_path.exists():
    history = json.loads(history_path.read_text())
    print("Detector minutes per epoch (train only | with validation):",
          [(round(h["train_minutes"], 2), round(h["minutes"], 2)) for h in history])
    best = max(history, key=lambda h: h["soft_dice"])
    print(f"Best detector epoch {best['epoch']}: val soft Dice {best['soft_dice']:.4f}, PQ {best['pq']:.4f}, "
          f"{best['predictions']:.1f} masks per photo at score 0.6")


## Crop refiner (10 epochs)

In [ ]:
from torch.utils.data import DataLoader

coco = COCO(str(ANNOTATIONS))
crop_paths = {}
for name, rows in [("train", train_rows), ("val", val_rows)]:
    crop_paths[name] = lib.build_crop_cache(coco, rows, TRAIN_DIR, WORK_DIR, name)
    print(f"{name} crops: {crop_paths[name][2]:,}")
loader_options = dict(num_workers=2, pin_memory=torch.cuda.is_available())
train_loader = DataLoader(lib.CropDataset(*crop_paths["train"][:2], augment=True), batch_size=CFG["refiner_batch"],
                          shuffle=True, drop_last=True, **loader_options)
val_loader = DataLoader(lib.CropDataset(*crop_paths["val"][:2]), batch_size=32, shuffle=False, **loader_options)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
started = time.monotonic()
refiner_history = lib.train_refiner(lib.CropUNet(pretrained=True), train_loader, val_loader,
                                    CFG["refiner_epochs"], WORK_DIR / "best_refiner.pt", device)
(WORK_DIR / "refiner_history.json").write_text(json.dumps(refiner_history, indent=2))
print(f"Refiner: {(time.monotonic() - started) / 60:.1f} min for {CFG['refiner_epochs']} epochs")


## Voting and tuning on the validation photos
Up to 10 candidates per view, 4 views, vote down to the cap. Compare the caps in the printout.

In [ ]:
def load_models():
    detector = lib.build_detector(pretrained=False).to(device)
    detector.load_state_dict(torch.load(WORK_DIR / "best_detector.pt", map_location=device, weights_only=False)["model"])
    refiner = lib.CropUNet(pretrained=False).to(device)
    refiner.load_state_dict(torch.load(WORK_DIR / "best_refiner.pt", map_location=device, weights_only=False)["model"])
    return detector.eval(), refiner.eval()


detector, refiner = load_models()
settings = dict(candidate_threshold=CFG["candidate_threshold"], max_candidates=CFG["max_candidates"])
groups = lib.all_annotator_ids(ANNOTATIONS)
references = {r["observation_id"]: [lib.annotation_rles(coco, rid, 2048) for rid in groups[r["observation_id"]]]
              for r in val_rows}
started = time.monotonic()
items = {}
for index, row in enumerate(val_rows, 1):
    items[row["observation_id"]] = lib.image_candidates(detector, refiner, TRAIN_DIR / row["file_name"], device, settings)
    if index % 25 == 0:
        print(f"  candidates for {index}/{len(val_rows)} validation photos "
              f"({(time.monotonic() - started) / 60:.1f} min)", flush=True)
counts = [len(item["candidates"]) for item in items.values()]
print(f"Candidate clusters per photo (up to {CFG['max_candidates']} per view, 4 views): mean {np.mean(counts):.1f}, "
      f"median {np.median(counts):.0f}, max {max(counts)}")
print("Photos with supported-by-all-4-views clusters, mean per photo:",
      round(np.mean([sum(c["support"] == 4 for c in item["candidates"]) for item in items.values()]), 2))

def blend(r):
    return (r["soft_dice"] + r["pq"]) / 2 if CFG["select_metric"] == "blend" else r[CFG["select_metric"]]


rows = []
baseline = {name: lib.single_view_rles(item, 0.6) for name, item in items.items()}
base = lib.mean_scores(baseline, references)
print("REFERENCE single view, detector masks only, score >= 0.6:", {k: round(v, 4) for k, v in base.items()})
recipe_rows = []
for threshold in CFG["refiner_thresholds"]:  # the public 0.70 recipe: identity view, score >= 0.7, refiner, merge
    predictions = {name: lib.select_instances(item, None, 1, threshold, min_score=0.7, view0_only=True)
                   for name, item in items.items()}
    scores = lib.mean_scores(predictions, references)
    recipe_rows.append(dict(cap=None, min_support=1, min_score=0.7, view0_only=True,
                            refiner_threshold=threshold, **scores))
for cap in CFG["caps"]:
    for support in CFG["min_supports"]:
        for floor in CFG["min_scores"]:
            for threshold in CFG["refiner_thresholds"]:
                predictions = {name: lib.select_instances(item, cap, support, threshold, min_score=floor)
                               for name, item in items.items()}
                scores = lib.mean_scores(predictions, references)
                rows.append(dict(cap=cap, min_support=support, min_score=floor, view0_only=False,
                                 refiner_threshold=threshold, **scores))
rows.sort(key=blend, reverse=True)
recipe_rows.sort(key=blend, reverse=True)
header = f"{'cap':>5} {'sup':>4} {'floor':>6} {'thr':>5} {'softDice':>9} {'PQ':>7} {'inst/img':>9}"
line = lambda r: (f"{str(r['cap']):>5} {r['min_support']:>4} {r['min_score']:>6} {r['refiner_threshold']:>5} "
                  f"{r['soft_dice']:>9.4f} {r['pq']:>7.4f} {r['predictions']:>9.2f}")
print("
PUBLIC RECIPE (identity view, score >= 0.7, no voting):")
print(header)
for r in recipe_rows:
    print(line(r))
print(f"
Voted settings, top 12 by {CFG['select_metric']} (validation photos; optimistic):")
print(header)
for r in rows[:12]:
    print(line(r))
print("
Best voted setting of each cap:")
for cap in CFG["caps"]:
    best = max((r for r in rows if r["cap"] == cap), key=blend)
    print(f"  cap={str(cap):>4}: softDice {best['soft_dice']:.4f}  PQ {best['pq']:.4f}  inst/img {best['predictions']:.2f} "
          f"(support {best['min_support']}, floor {best['min_score']}, threshold {best['refiner_threshold']})")
CHOSEN = rows[0] if blend(rows[0]) >= blend(recipe_rows[0]) + CFG["recipe_margin"] else recipe_rows[0]
print("
Using the", "voted setting" if CHOSEN is rows[0] else "public recipe",
      f"(margin needed over the recipe: {CFG['recipe_margin']})")
(WORK_DIR / "tuning_grid.json").write_text(json.dumps(dict(reference=base, chosen=CHOSEN, recipe=recipe_rows, grid=rows), indent=2))
print("\nCHOSEN:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in CHOSEN.items()})


## Test predictions and submission file
Nothing is uploaded: the last cell prints the submit command.

In [ ]:
started = time.monotonic()
test_predictions, fallback_images = {}, []
for index, path in enumerate(test_files, 1):
    item = lib.image_candidates(detector, refiner, path, device, settings)
    rles = lib.select_instances(item, CHOSEN["cap"], CHOSEN["min_support"], CHOSEN["refiner_threshold"],
                                min_score=CHOSEN["min_score"], view0_only=CHOSEN["view0_only"])
    if not rles:
        fallback_images.append(path.stem)
    test_predictions[path.stem] = rles
    if index % 30 == 0:
        print(f"  predicted {index}/{len(test_files)} test photos ({(time.monotonic() - started) / 60:.1f} min)", flush=True)

import csv

submission_rows = []
for observation_id, rles in test_predictions.items():
    for number, rle in enumerate(rles, 1):
        counts = rle["counts"].decode("ascii") if isinstance(rle["counts"], bytes) else rle["counts"]
        submission_rows.append((f"{observation_id}_{number}", counts))
with open(SUBMISSION_PATH, "w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(["filament_id", "segmentation_rle"])
    writer.writerows(submission_rows)
per_image = np.array([len(v) for v in test_predictions.values()])
overlaps = sum(
    1 for rles in test_predictions.values() if len(rles) > 1
    and (np.tril(mask_utils.iou(rles, rles, [0] * len(rles)), -1) > 0).any())
print(f"Test predictions took {(time.monotonic() - started) / 60:.1f} min")
print(f"Rows: {len(submission_rows):,} | photos covered: {len(test_predictions)}/{len(test_files)} | unique ids: "
      f"{len({r[0] for r in submission_rows}) == len(submission_rows)}")
print(f"Instances per photo: mean {per_image.mean():.2f}, median {np.median(per_image):.0f}, min {per_image.min()}, "
      f"max {per_image.max()} | photos with overlapping masks: {overlaps} | photos with no mask: {len(fallback_images)}")
if fallback_images:
    print("WARNING: no detection for", fallback_images[:10])


In [ ]:
print("Submission file:", SUBMISSION_PATH)
print("Submit with: kaggle competitions submit -c", COMPETITION, "-f", SUBMISSION_PATH, "-m",
      "V14 Mask R-CNN + ResNet18 refiner, 15 epochs, " + ("recipe" if CHOSEN["view0_only"] else "voted cap " + str(CHOSEN["cap"])))
